# CareerMatch AI - Notebook 01: Vector Index Setup

**Purpose:** Create a Delta Sync vector search index on job descriptions to enable semantic job matching.

**What this notebook does:**
1. Configures the data source and catalog/schema names
2. Validates the source table exists and has expected columns
3. Creates a vector search endpoint (if not exists)
4. Creates a Delta Sync vector index on the `description` column
5. Verifies the index is ready for queries

**Prerequisites:**
- Access to Unity Catalog
- Permissions to create vector search endpoints and indexes
- Source table with job postings data

## Install Dependencies

In [0]:
%pip install databricks-vectorsearch --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

## Configuration

**ACTION REQUIRED:** Update these values to match your environment.

In [0]:
# =============================================================================
# DATA SOURCE CONFIGURATION
# =============================================================================

CATALOG = "finalproject"
SCHEMA = "careermatch_ai"

# Full agent-ready table created by Notebook 00
BASE_JOBS_TABLE_PATH = f"{CATALOG}.{SCHEMA}.jobs_vector_ready"

# Smaller engineering-only table created in this notebook
JOBS_TABLE = "jobs_engineering"
JOBS_TABLE_PATH = f"{CATALOG}.{SCHEMA}.{JOBS_TABLE}"

# =============================================================================
# VECTOR SEARCH CONFIGURATION
# =============================================================================

VECTOR_SEARCH_ENDPOINT = "careermatch_endpoint_gerek"

# New name prevents collision with the full-table index
VECTOR_INDEX_NAME = f"{CATALOG}.{SCHEMA}.jobs_engineering_search_index"

PRIMARY_KEY_COLUMN = "job_id"
EMBEDDING_SOURCE_COLUMN = "search_text"
EMBEDDING_MODEL_ENDPOINT = "databricks-bge-large-en"

print(f"Base table: {BASE_JOBS_TABLE_PATH}")
print(f"Engineering table: {JOBS_TABLE_PATH}")
print(f"Vector index: {VECTOR_INDEX_NAME}")

Base table: finalproject.careermatch_ai.jobs_vector_ready
Engineering table: finalproject.careermatch_ai.jobs_engineering
Vector index: finalproject.careermatch_ai.jobs_engineering_search_index


In [0]:
# Create an engineering-only subset from the final table produced by Notebook 00.

from pyspark.sql import functions as F

base_df = spark.table(BASE_JOBS_TABLE_PATH)

engineering_df = (
    base_df
    .filter(
        F.col("title").isNotNull()
        & F.col("title").rlike("(?i)engineer")
    )
    .dropDuplicates(["job_id"])
)

engineering_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(JOBS_TABLE_PATH)

engineering_count = spark.table(JOBS_TABLE_PATH).count()

print(f"✓ Created engineering table: {JOBS_TABLE_PATH}")
print(f"  Engineering jobs: {engineering_count:,}")

display(
    spark.table(JOBS_TABLE_PATH)
    .select(
        "job_id",
        "title",
        "company_name",
        "location",
        "description",
        "search_text"
    )
    .limit(5)
)

✓ Created engineering table: finalproject.careermatch_ai.jobs_engineering
  Engineering jobs: 11,072


job_id title company_name location description search_text 3898168752 Industrial Engineer Ingersoll Rand Enon, OH Job Title

Industrial Engineer

Location

Enon, OH

About Us

With a passion for technology and solutions, our SEEPEX division is a leading worldwide specialist. We’re pushing the boundaries with progressive cavity pumps, advanced systems, and digital solutions. From the depths of the sea floor to pumps that keep breweries pouring sustainability all year long, we keep everything flowing.

Job Summary

If you’re ready to help us reshape the manufacturing landscape, we’re seeking an Industrial Engineer with the seasoned expertise in tooling design coupled with a proven ability in refining manufacturing processes.

You’ll serve as a vital partner to our assembly and manufacturing teams as you collaborate on various issues and be the go-to problem solver. Armed with insight, you'll design jigs and fixtures that streamlines manufacturing operations and enhances worker efficiency and safety at every step.

Bringing knowledge in rubber injection, plastic modeling, and machining is preferred as you lead efforts to improve throughput, profitability, functionality, and strategic competitive advantage while reducing scrap. If you thrive in developing solutions and in making an impactful difference, we want you on our team.

Responsibilities

Tooling and Mold needs including improvements, building up, purchase requisitions, trials, maintaining tooling quality, and core sizing. Incorporates tooling repair, tooling replacement, new tooling, and new stator models.Assist with long-range and short-term capital needs including work scopes, layouts, budgets, justifications, Capital Request forms, purchase requisitions, trials, and maintaining equipment quality. Involvement may be researching, supporting other groups‘ efforts, or assisting the implementation. For large machinery, locate, develop, and manage suppliers as needed to define and meet technical requirements.Rubber injection molding manufacturing processes: Trials, implementation, help determine best practices, improvements, optimization, improving product quality, reducing scrap, eliminating ROT steps, improving sustainability, determining safe material handling, and auditing operators for adherence to requirements.Scrap reduction including ownership of assigned non-conformance issues, implementing proven robust corrective actions, developing, and conducting studies to address problems.Create and modify stator CAD models and tooling drawings, ensuring SEEPEX drawing standards and storage standards are met.Lean, ISO, Quality and Safety initiatives

Basic Qualifications

A minimum of 5-10 years of progressive experience in Manufacturing Engineering.Bachelor's degree from a four-year college or accredited university (in Engineering preferred)Experience in tooling design and developing manufacturing processes.Effective written & verbal communication skillsAt minimum must be a lawful Permanent Resident of the US, as this role is not eligible for Visa Sponsorship

Travel & Work Arrangements / Requirements

Working onsite in out Enon, OH facilityThis role is not eligible for Visa Sponsorship

Key Competencies 

Experience with the principles of 5S, Lean, Kaizen, or similar.Experience in tooling design, rubber injection molding, and developing manufacturing processes.Experience/demonstrated proficiency with CAD applications.Excellent oral and written communication skills demonstrated by the ability to communicate across all levels within and outside of the organization and in cross-function collaborations.Ability to present facts and recommendations, to present opinions, and to defend them.

What We Offer:

Benefits

Equity: All Employee Grant401k plan with a company match Medical and Prescription drug plansWellness and Chronic disease management programsDental, vision, life/AD&D insuranceShort- and Long-term disabilityHealth Savings AccountFlexible Spending AccountParental LeaveEm

In [0]:
source_df = spark.table(JOBS_TABLE_PATH)

print(f"Source table: {JOBS_TABLE_PATH}")
print(f"Total records: {source_df.count():,}")
print(f"Columns: {len(source_df.columns)}")
print(f"Available columns: {source_df.columns}")

display(
    source_df.select(
        "job_id",
        "title",
        "company_name",
        "location",
        "formatted_work_type",
        "remote_status",
        "description",
        "search_text"
    ).limit(5)
)

Source table: finalproject.careermatch_ai.jobs_engineering
Total records: 11,072
Columns: 11
Available columns: ['job_id', 'title', 'company_name', 'location', 'formatted_work_type', 'remote_status', 'salary_category', 'normalized_salary', 'description', 'search_text', 'job_posting_url']


job_id title company_name location formatted_work_type remote_status description search_text 3898168752 Industrial Engineer Ingersoll Rand Enon, OH Full-time Not Remote Job Title

Industrial Engineer

Location

Enon, OH

About Us

With a passion for technology and solutions, our SEEPEX division is a leading worldwide specialist. We’re pushing the boundaries with progressive cavity pumps, advanced systems, and digital solutions. From the depths of the sea floor to pumps that keep breweries pouring sustainability all year long, we keep everything flowing.

Job Summary

If you’re ready to help us reshape the manufacturing landscape, we’re seeking an Industrial Engineer with the seasoned expertise in tooling design coupled with a proven ability in refining manufacturing processes.

You’ll serve as a vital partner to our assembly and manufacturing teams as you collaborate on various issues and be the go-to problem solver. Armed with insight, you'll design jigs and fixtures that streamlines manufacturing operations and enhances worker efficiency and safety at every step.

Bringing knowledge in rubber injection, plastic modeling, and machining is preferred as you lead efforts to improve throughput, profitability, functionality, and strategic competitive advantage while reducing scrap. If you thrive in developing solutions and in making an impactful difference, we want you on our team.

Responsibilities

Tooling and Mold needs including improvements, building up, purchase requisitions, trials, maintaining tooling quality, and core sizing. Incorporates tooling repair, tooling replacement, new tooling, and new stator models.Assist with long-range and short-term capital needs including work scopes, layouts, budgets, justifications, Capital Request forms, purchase requisitions, trials, and maintaining equipment quality. Involvement may be researching, supporting other groups‘ efforts, or assisting the implementation. For large machinery, locate, develop, and manage suppliers as needed to define and meet technical requirements.Rubber injection molding manufacturing processes: Trials, implementation, help determine best practices, improvements, optimization, improving product quality, reducing scrap, eliminating ROT steps, improving sustainability, determining safe material handling, and auditing operators for adherence to requirements.Scrap reduction including ownership of assigned non-conformance issues, implementing proven robust corrective actions, developing, and conducting studies to address problems.Create and modify stator CAD models and tooling drawings, ensuring SEEPEX drawing standards and storage standards are met.Lean, ISO, Quality and Safety initiatives

Basic Qualifications

A minimum of 5-10 years of progressive experience in Manufacturing Engineering.Bachelor's degree from a four-year college or accredited university (in Engineering preferred)Experience in tooling design and developing manufacturing processes.Effective written & verbal communication skillsAt minimum must be a lawful Permanent Resident of the US, as this role is not eligible for Visa Sponsorship

Travel & Work Arrangements / Requirements

Working onsite in out Enon, OH facilityThis role is not eligible for Visa Sponsorship

Key Competencies 

Experience with the principles of 5S, Lean, Kaizen, or similar.Experience in tooling design, rubber injection molding, and developing manufacturing processes.Experience/demonstrated proficiency with CAD applications.Excellent oral and written communication skills demonstrated by the ability to communicate across all levels within and outside of the organization and in cross-function collaborations.Ability to present facts and recommendations, to present opinions, and to defend them.

What We Offer:

Benefits

Equity: All Employee Grant401k plan with a company match Medical and Prescription drug plansWellness and Chronic disease management programsDental, vision, life/AD&D insuranceShort- and Long-term disabilityHealth S

## Imports and Client Setup

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.vector_search.client import VectorSearchClient
import time

# Initialize clients
w = WorkspaceClient()
vsc = VectorSearchClient()

print("✓ Clients initialized")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
✓ Clients initialized


## Step 1: Validate Source Table

Verify the source table exists and contains the expected columns.

In [0]:
# Check table exists and get schema
# Validate the source table created by Notebook 00

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

try:
    df = spark.table(JOBS_TABLE_PATH)
    columns = df.columns
    row_count = df.count()

    print(f"✓ Table '{JOBS_TABLE_PATH}' found")
    print(f"  Columns: {len(columns)}")
    print(f"  Total rows: {row_count:,}")

    required_columns = [
        PRIMARY_KEY_COLUMN,
        EMBEDDING_SOURCE_COLUMN,
        "title",
        "description"
    ]

    missing_columns = [
        column_name
        for column_name in required_columns
        if column_name not in columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    print(f"✓ Required columns present: {required_columns}")

    primary_key_type = df.schema[PRIMARY_KEY_COLUMN].dataType
    source_text_type = df.schema[EMBEDDING_SOURCE_COLUMN].dataType

    print(f"  Primary key type: {primary_key_type}")
    print(f"  Embedding source type: {source_text_type}")

    if not isinstance(source_text_type, StringType):
        raise TypeError(
            f"{EMBEDDING_SOURCE_COLUMN} must be a string column, "
            f"but found {source_text_type}"
        )

    null_key_count = df.filter(
        F.col(PRIMARY_KEY_COLUMN).isNull()
    ).count()

    null_text_count = df.filter(
        F.col(EMBEDDING_SOURCE_COLUMN).isNull()
        | (F.length(F.trim(F.col(EMBEDDING_SOURCE_COLUMN))) == 0)
    ).count()

    duplicate_key_count = (
        df.groupBy(PRIMARY_KEY_COLUMN)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    print(f"  Null primary keys: {null_key_count:,}")
    print(f"  Empty search_text values: {null_text_count:,}")
    print(f"  Duplicate primary keys: {duplicate_key_count:,}")

    if null_key_count > 0:
        raise ValueError("Primary key contains null values")

    if duplicate_key_count > 0:
        raise ValueError("Primary key contains duplicate values")

    if null_text_count > 0:
        raise ValueError(
            f"{EMBEDDING_SOURCE_COLUMN} contains null or empty values"
        )

    print("✓ Source table passed validation")

except Exception as e:
    print(f"✗ Error: {e}")
    raise

✓ Table 'finalproject.careermatch_ai.jobs_engineering' found
  Columns: 11
  Total rows: 11,072
✓ Required columns present: ['job_id', 'search_text', 'title', 'description']
  Primary key type: LongType()
  Embedding source type: StringType()
  Null primary keys: 0
  Empty search_text values: 0
  Duplicate primary keys: 0
✓ Source table passed validation


In [0]:
# Standard Vector Search endpoints require Change Data Feed on the source table.

spark.sql(f"""
    ALTER TABLE {JOBS_TABLE_PATH}
    SET TBLPROPERTIES (
        delta.enableChangeDataFeed = true
    )
""")

table_properties = spark.sql(
    f"SHOW TBLPROPERTIES {JOBS_TABLE_PATH}"
)

cdf_setting = (
    table_properties
    .filter("key = 'delta.enableChangeDataFeed'")
    .collect()
)

if cdf_setting and cdf_setting[0]["value"].lower() == "true":
    print(f"✓ Change Data Feed enabled on {JOBS_TABLE_PATH}")
else:
    raise RuntimeError(
        f"Change Data Feed could not be verified on {JOBS_TABLE_PATH}"
    )

✓ Change Data Feed enabled on finalproject.careermatch_ai.jobs_engineering


## Source Table Validation Complete

The source table was successfully validated. It contains the expected primary key and search text column, and Change Data Feed is enabled.

In [0]:
display(
    spark.table(JOBS_TABLE_PATH)
    .select(
        "job_id",
        "title",
        "company_name",
        "location",
        "formatted_work_type",
        "remote_status",
        "search_text"
    )
    .limit(5)
)

job_id title company_name location formatted_work_type remote_status search_text 3898168752 Industrial Engineer Ingersoll Rand Enon, OH Full-time Not Remote Industrial Engineer Ingersoll Rand Enon, OH Full-time Not Remote Not Listed Job Title

Industrial Engineer

Location

Enon, OH

About Us

With a passion for technology and solutions, our SEEPEX division is a leading worldwide specialist. We’re pushing the boundaries with progressive cavity pumps, advanced systems, and digital solutions. From the depths of the sea floor to pumps that keep breweries pouring sustainability all year long, we keep everything flowing.

Job Summary

If you’re ready to help us reshape the manufacturing landscape, we’re seeking an Industrial Engineer with the seasoned expertise in tooling design coupled with a proven ability in refining manufacturing processes.

You’ll serve as a vital partner to our assembly and manufacturing teams as you collaborate on various issues and be the go-to problem solver. Armed with insight, you'll design jigs and fixtures that streamlines manufacturing operations and enhances worker efficiency and safety at every step.

Bringing knowledge in rubber injection, plastic modeling, and machining is preferred as you lead efforts to improve throughput, profitability, functionality, and strategic competitive advantage while reducing scrap. If you thrive in developing solutions and in making an impactful difference, we want you on our team.

Responsibilities

Tooling and Mold needs including improvements, building up, purchase requisitions, trials, maintaining tooling quality, and core sizing. Incorporates tooling repair, tooling replacement, new tooling, and new stator models.Assist with long-range and short-term capital needs including work scopes, layouts, budgets, justifications, Capital Request forms, purchase requisitions, trials, and maintaining equipment quality. Involvement may be researching, supporting other groups‘ efforts, or assisting the implementation. For large machinery, locate, develop, and manage suppliers as needed to define and meet technical requirements.Rubber injection molding manufacturing processes: Trials, implementation, help determine best practices, improvements, optimization, improving product quality, reducing scrap, eliminating ROT steps, improving sustainability, determining safe material handling, and auditing operators for adherence to requirements.Scrap reduction including ownership of assigned non-conformance issues, implementing proven robust corrective actions, developing, and conducting studies to address problems.Create and modify stator CAD models and tooling drawings, ensuring SEEPEX drawing standards and storage standards are met.Lean, ISO, Quality and Safety initiatives

Basic Qualifications

A minimum of 5-10 years of progressive experience in Manufacturing Engineering.Bachelor's degree from a four-year college or accredited university (in Engineering preferred)Experience in tooling design and developing manufacturing processes.Effective written & verbal communication skillsAt minimum must be a lawful Permanent Resident of the US, as this role is not eligible for Visa Sponsorship

Travel & Work Arrangements / Requirements

Working onsite in out Enon, OH facilityThis role is not eligible for Visa Sponsorship

Key Competencies 

Experience with the principles of 5S, Lean, Kaizen, or similar.Experience in tooling design, rubber injection molding, and developing manufacturing processes.Experience/demonstrated proficiency with CAD applications.Excellent oral and written communication skills demonstrated by the ability to communicate across all levels within and outside of the organization and in cross-function collaborations.Ability to present facts and recommendations, to present opinions, and to defend them.

What We Offer:

Benefits

Equity: All Employee Grant401k plan with a company match Medical and Prescription drug plansWellness and Chronic disease management programsDental, vi

## Step 2: Create Vector Search Endpoint

The endpoint is the compute resource that hosts vector indexes.
One endpoint can host multiple indexes.

In [0]:
def get_or_create_endpoint(endpoint_name: str):
    """Return an existing Vector Search endpoint or create it."""

    try:
        endpoint = vsc.get_endpoint(endpoint_name)

        print(f"✓ Endpoint '{endpoint_name}' already exists")
        print(
            "  Status:",
            endpoint.get("endpoint_status", {}).get(
                "state",
                "UNKNOWN"
            )
        )

        return endpoint

    except Exception as e:
        error_text = str(e).lower()

        not_found_markers = [
            "not found",
            "does not exist",
            "resource_does_not_exist",
            "not_found"
        ]

        if not any(
            marker in error_text
            for marker in not_found_markers
        ):
            raise

    print(f"Creating endpoint '{endpoint_name}'...")

    endpoint = vsc.create_endpoint(
        name=endpoint_name,
        endpoint_type="STANDARD"
    )

    print("✓ Endpoint creation initiated")
    return endpoint


endpoint = get_or_create_endpoint(
    VECTOR_SEARCH_ENDPOINT
)

✓ Endpoint 'careermatch_endpoint_gerek' already exists
  Status: ONLINE


In [0]:
def wait_for_endpoint(endpoint_name: str, timeout_minutes: int = 20):
    """Wait for endpoint to reach ONLINE state."""

    start_time = time.time()
    timeout_seconds = timeout_minutes * 60

    while time.time() - start_time < timeout_seconds:
        endpoint = vsc.get_endpoint(endpoint_name)
        state = endpoint.get("endpoint_status", {}).get("state", "UNKNOWN")

        if state == "ONLINE":
            print(f"✓ Endpoint '{endpoint_name}' is ONLINE")
            return endpoint

        print(f"  Endpoint state: {state} (waiting...)")
        time.sleep(30)

    raise TimeoutError(f"Endpoint did not come online within {timeout_minutes} minutes")


endpoint = wait_for_endpoint(VECTOR_SEARCH_ENDPOINT)

✓ Endpoint 'careermatch_endpoint_gerek' is ONLINE


## Step 3: Create Delta Sync Vector Index

Notebook 00 created a text column named `search_text`, but it did not create precomputed embedding vectors.

This notebook therefore uses a Delta Sync index with Databricks-managed embeddings. Databricks will generate embeddings from `search_text` and synchronize the source Delta table with the Vector Search index.

Triggered synchronization is used so updates can be started manually when the source table changes.

In [0]:
def get_or_create_delta_sync_index(
    endpoint_name: str,
    index_name: str,
    source_table: str,
    primary_key: str,
    embedding_source_column: str,
    embedding_model_endpoint: str
):
    """Return an existing Delta Sync index or create a new one."""

    try:
        index = vsc.get_index(
            endpoint_name=endpoint_name,
            index_name=index_name
        )

        print(f"✓ Index '{index_name}' already exists")
        return index

    except Exception as e:
        error_text = str(e).lower()

        not_found_markers = [
            "not found",
            "does not exist",
            "resource_does_not_exist",
            "not_found"
        ]

        if not any(
            marker in error_text
            for marker in not_found_markers
        ):
            raise

    print(f"Creating Delta Sync index '{index_name}'...")
    print(f"  Source table: {source_table}")
    print(f"  Primary key: {primary_key}")
    print(f"  Embedding source: {embedding_source_column}")
    print(f"  Embedding model: {embedding_model_endpoint}")

    index = vsc.create_delta_sync_index(
        endpoint_name=endpoint_name,
        index_name=index_name,
        source_table_name=source_table,
        pipeline_type="TRIGGERED",
        primary_key=primary_key,
        embedding_source_column=embedding_source_column,
        embedding_model_endpoint_name=embedding_model_endpoint
    )

    print("✓ Delta Sync index creation initiated")
    return index


index = get_or_create_delta_sync_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=VECTOR_INDEX_NAME,
    source_table=JOBS_TABLE_PATH,
    primary_key=PRIMARY_KEY_COLUMN,
    embedding_source_column=EMBEDDING_SOURCE_COLUMN,
    embedding_model_endpoint=EMBEDDING_MODEL_ENDPOINT
)

Creating Delta Sync index 'finalproject.careermatch_ai.jobs_engineering_search_index'...
  Source table: finalproject.careermatch_ai.jobs_engineering
  Primary key: job_id
  Embedding source: search_text
  Embedding model: databricks-bge-large-en
✓ Delta Sync index creation initiated


In [0]:
# This timed out. I changed timeout_minutes to 1440 from 360. Although it failed the server side indexing pipline likely still made progress on the whole thing. We can delete this comment later.

def wait_for_index(
    endpoint_name: str,
    index_name: str,
    timeout_minutes: int = 1440
):
    """Wait for the Delta Sync index to become ready."""

    start_time = time.time()
    timeout_seconds = timeout_minutes * 60

    while time.time() - start_time < timeout_seconds:
        try:
            index = vsc.get_index(
                endpoint_name=endpoint_name,
                index_name=index_name
            )

            index_description = index.describe()
            status = index_description.get("status", {})

            detailed_state = status.get(
                "detailed_state",
                "UNKNOWN"
            )

            ready = status.get("ready", False)

            indexed_count = status.get(
                "indexed_row_count",
                "N/A"
            )

            print(
                f"Index state: {detailed_state} | "
                f"Indexed rows: {indexed_count}"
            )

            if ready:
                print(f"✓ Index '{index_name}' is READY")
                return index

        except Exception as e:
            print(f"Status check: {e}")

        time.sleep(30)

    raise TimeoutError(
        f"Index did not become ready within "
        f"{timeout_minutes} minutes"
    )


index = wait_for_index(
    VECTOR_SEARCH_ENDPOINT,
    VECTOR_INDEX_NAME
)

Index state: ONLINE_NO_PENDING_UPDATE | Indexed rows: 11072
✓ Index 'finalproject.careermatch_ai.jobs_engineering_search_index' is READY


In [0]:
index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=VECTOR_INDEX_NAME
)

details = index.describe()

print(details)

{'name': 'finalproject.careermatch_ai.jobs_engineering_search_index', 'endpoint_name': 'careermatch_endpoint_gerek', 'primary_key': 'job_id', 'index_type': 'DELTA_SYNC', 'delta_sync_index_spec': {'source_table': 'finalproject.careermatch_ai.jobs_engineering', 'embedding_source_columns': [{'name': 'search_text', 'embedding_model_endpoint_name': 'databricks-bge-large-en'}], 'pipeline_type': 'TRIGGERED', 'pipeline_id': '6d54c4fb-ab18-42f0-b767-2b6fba6a8392'}, 'status': {'detailed_state': 'ONLINE_NO_PENDING_UPDATE', 'message': 'Index creation succeeded. Check latest status: https://dbc-c31d04be-d8a2.cloud.databricks.com/explore/data/finalproject/careermatch_ai/jobs_engineering_search_index', 'indexed_row_count': 11072, 'triggered_update_status': {'last_processed_commit_version': 6, 'last_processed_commit_timestamp': '2026-06-14T10:56:09Z'}, 'ready': True, 'index_url': 'dbc-c31d04be-d8a2.cloud.databricks.com/api/2.0/vector-search/indexes/finalproject.careermatch_ai.jobs_engineering_search_i

In [0]:
index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=VECTOR_INDEX_NAME
)

index_description = index.describe()
status = index_description.get("status", {})

print(f"Index name: {VECTOR_INDEX_NAME}")
print(f"State: {status.get('detailed_state', 'UNKNOWN')}")
print(f"Indexed rows: {status.get('indexed_row_count', 'N/A')}")
print(f"Ready: {status.get('ready', False)}")

Index name: finalproject.careermatch_ai.jobs_engineering_search_index
State: ONLINE_NO_PENDING_UPDATE
Indexed rows: 11072
Ready: True


## Vector Search Index Validation

The Vector Search index was successfully created and connected to the engineering jobs source table.

Validation checks completed:

- The index reached a ready/online state.
- The source table was available and populated.
- The index was configured with the selected embedding endpoint.
- The indexed data includes the engineering job records required by the agent.
- The index can now be used by later notebooks for semantic job retrieval.

This confirms that the vector-search component is ready for agent testing.

## Step 4: Test the Vector Index

Run a sample query to verify the index works correctly.

In [0]:
index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=VECTOR_INDEX_NAME
)

test_query = (
    "machine learning engineer with Python, "
    "TensorFlow, and production ML experience"
)

result_columns = [
    PRIMARY_KEY_COLUMN,
    "title",
    "company_name",
    "location",
    "description"
]

results = index.similarity_search(
    query_text=test_query,
    columns=result_columns,
    num_results=5
)

rows = results.get(
    "result",
    {}
).get(
    "data_array",
    []
)

print(f"Query: {test_query}")
print(f"Results returned: {len(rows)}\n")

if not rows:
    print("⚠ No results were returned.")
else:
    for position, row in enumerate(rows, start=1):
        job_id = row[0]
        title = row[1] or "Title unavailable"
        company = row[2] or "Company unavailable"
        location = row[3] or "Location unavailable"

        print(f"{position}. {title} at {company}")
        print(f"   Location: {location}")
        print(f"   Job ID: {job_id}")
        print()

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Query: machine learning engineer with Python, TensorFlow, and production ML experience
Results returned: 5

1. Senior Machine Learning Engineer at Square One Resources
   Location: California, United States
   Job ID: 3904572433.0

2. Senior Machine Learning Research Engineer at Symbolica AI
   Location: San Francisco Bay Area
   Job ID: 3848960304.0

3. Machine Learning Engineer- 10+year exp* at Q1 Technologies, Inc.
   Location: Sunnyvale, CA
   Job ID: 3901394474.0

4. Machine Learning Engineer at Indotronix Avani Group
   Location: Pennsylvania, United States
   Job ID: 3902371576.0

5. Senior Machine Learning Engineer at Stefanini North America and APAC
   Location: Dearborn, MI
   Job ID: 3893246261.0



## Semantic Search Results Review

The test query returned engineering and machine-learning-related job postings from the Vector Search index.

The results were reviewed for:

- Relevance to the search query
- Appropriate job titles and descriptions
- Engineering-focused content
- Successful retrieval from the indexed jobs table

The returned records demonstrate that the index can retrieve semantically related job postings rather than relying only on exact keyword matches. This search functionality will be used by the CareerMatch AI agent to identify relevant opportunities for users.

## Step 5: Save Configuration for Next Notebook

Store the configuration values so subsequent notebooks can use them.

In [0]:
config_rows = [
    ("CATALOG", CATALOG),
    ("SCHEMA", SCHEMA),
    ("JOBS_TABLE", JOBS_TABLE),
    ("JOBS_TABLE_PATH", JOBS_TABLE_PATH),
    ("VECTOR_SEARCH_ENDPOINT", VECTOR_SEARCH_ENDPOINT),
    ("VECTOR_INDEX_NAME", VECTOR_INDEX_NAME),
    ("PRIMARY_KEY_COLUMN", PRIMARY_KEY_COLUMN),
    ("EMBEDDING_SOURCE_COLUMN", EMBEDDING_SOURCE_COLUMN),
    ("EMBEDDING_MODEL_ENDPOINT", EMBEDDING_MODEL_ENDPOINT),
]

config_df = spark.createDataFrame(
    config_rows,
    ["key", "value"]
)

CONFIG_TABLE = f"{CATALOG}.{SCHEMA}.careermatch_config"

config_df.write \
    .mode("overwrite") \
    .saveAsTable(CONFIG_TABLE)

print(f"✓ Configuration saved to '{CONFIG_TABLE}'")

✓ Configuration saved to 'finalproject.careermatch_ai.careermatch_config'


In [0]:
display(spark.table(CONFIG_TABLE))

key,value
CATALOG,finalproject
SCHEMA,careermatch_ai
JOBS_TABLE,jobs_engineering
JOBS_TABLE_PATH,finalproject.careermatch_ai.jobs_engineering
VECTOR_SEARCH_ENDPOINT,careermatch_endpoint_gerek
VECTOR_INDEX_NAME,finalproject.careermatch_ai.jobs_engineering_search_index
PRIMARY_KEY_COLUMN,job_id
EMBEDDING_SOURCE_COLUMN,search_text
EMBEDDING_MODEL_ENDPOINT,databricks-bge-large-en


## Summary

**Completed:**

1. Loaded the agent-ready table created by Notebook 00
2. Validated `job_id` and `search_text`
3. Enabled Change Data Feed on the source Delta table
4. Created or reused the Vector Search endpoint
5. Created a managed-embedding Delta Sync index
6. Tested semantic job retrieval
7. Saved the configuration for Notebook 02

**Source table:**  
`finalproject.careermatch_ai.jobs_vector_ready`

**Embedding source:**  
`search_text`

**Vector index:**  
`finalproject.careermatch_ai.jobs_search_text_index`

**Next:** Run Notebook 02 to create the Unity Catalog tools used by the CareerMatch agent.